# Phase 1 — Data Collection & Audit (Colab)

**Pre-requisite:** Run `00_colab_setup.ipynb` first to mount Google Drive.

This notebook:
1. Downloads ~400 robot images automatically using `icrawler` (Bing image search)
2. Deduplicates and filters them
3. Uploads the images to Roboflow for annotation in their free web UI
4. After you annotate, exports the dataset back in YOLO format
5. Runs mandatory VIZ 1.A / 1.B / 1.C audit checks

**Target:** ≥ 300 annotated images, ≥ 25 instances per class in train set.

---
### Workflow overview
```
Step 1: Run cells A1–A3  → download ~400 raw images to Drive
Step 2: Run cell B1      → upload images to Roboflow project
Step 3: Go to Roboflow   → annotate all images (arm/leg/torso/head/sensor)
Step 4: Run cell C1–C2   → export annotated dataset back to Drive
Step 5: Run cells D1–D3  → generate audit visualizations
```

In [ ]:
import os, sys, random, collections, shutil, hashlib
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_DIR = '/content/drive/MyDrive/robot-perception'
RAW_DIR     = f'{PROJECT_DIR}/data/raw'
DATA_DIR    = f'{PROJECT_DIR}/data/annotated'
RESULTS_DIR = f'{PROJECT_DIR}/results/figures'

for d in [RAW_DIR, RESULTS_DIR,
          f'{DATA_DIR}/images/train', f'{DATA_DIR}/images/val',
          f'{DATA_DIR}/images/test',  f'{DATA_DIR}/images/calibration',
          f'{DATA_DIR}/labels/train', f'{DATA_DIR}/labels/val',
          f'{DATA_DIR}/labels/test',  f'{DATA_DIR}/labels/calibration']:
    os.makedirs(d, exist_ok=True)

CLASS_NAMES = ['arm', 'leg', 'torso', 'head', 'sensor']
print('Directories ready.')

## Part A — Download raw robot images

We use `icrawler` to pull images from Bing across ~20 search queries covering diverse humanoid robots.
Each query fetches ~25 images → ~500 total before deduplication → keep best 350+.

**Runtime: ~5–10 minutes**

In [ ]:
# Cell A1 — Install icrawler
!pip install -q icrawler

In [ ]:
# Cell A2 — Download images across diverse robot search queries
from icrawler.builtin import BingImageCrawler

# ~25 images per query × 20 queries = ~500 raw images
SEARCH_QUERIES = [
    # Full-body bipedal humanoids (best for all 5 classes)
    'Boston Dynamics Atlas humanoid robot full body',
    'ASIMO Honda robot walking full body',
    'Unitree H1 humanoid robot',
    'Agility Robotics Digit robot',
    'Figure AI humanoid robot',
    'Fourier Intelligence humanoid robot',
    'UBTECH Walker humanoid robot',
    'Sanctuary AI Phoenix humanoid robot',
    # Specific body parts visible
    'humanoid robot arm gripper close up',
    'humanoid robot legs walking',
    'robot torso chest body',
    'robot head sensor camera',
    # Industrial/research robots with clear limbs
    'NAO robot humanoid full body',
    'Pepper robot Softbank full body',
    'Spot robot Boston Dynamics legs',
    'Valkyrie NASA humanoid robot',
    'iCub humanoid robot',
    'ROMEO robot humanoid',
    # More variety
    'humanoid robot laboratory research',
    'bipedal robot competition DARPA',
]

IMAGES_PER_QUERY = 25

for i, query in enumerate(SEARCH_QUERIES):
    # Use a safe folder name derived from query index
    out_dir = os.path.join(RAW_DIR, f'q{i:02d}')
    os.makedirs(out_dir, exist_ok=True)

    # Skip if already downloaded (re-run safe)
    existing = [f for f in os.listdir(out_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
    if len(existing) >= IMAGES_PER_QUERY * 0.8:
        print(f'[{i:02d}] Already have {len(existing)} images, skipping: {query[:50]}')
        continue

    print(f'[{i:02d}] Downloading: {query[:60]}')
    try:
        crawler = BingImageCrawler(storage={'root_dir': out_dir})
        crawler.crawl(keyword=query, max_num=IMAGES_PER_QUERY, min_size=(200, 200))
        downloaded = len([f for f in os.listdir(out_dir) if f.endswith(('.jpg', '.png', '.jpeg'))])
        print(f'      → {downloaded} images saved')
    except Exception as e:
        print(f'      ERROR: {e}')

total = sum(
    len([f for f in os.listdir(os.path.join(RAW_DIR, d)) if f.endswith(('.jpg', '.png', '.jpeg'))])
    for d in os.listdir(RAW_DIR) if os.path.isdir(os.path.join(RAW_DIR, d))
)
print(f'\nTotal raw images downloaded: {total}')

# Cell A3 — Deduplicate by MD5 hash + filter corrupt/tiny images
# Flattens all query subdirs into RAW_DIR/all_unique/

ALL_UNIQUE_DIR = os.path.join(RAW_DIR, 'all_unique')
os.makedirs(ALL_UNIQUE_DIR, exist_ok=True)

seen_hashes = set()
kept, skipped_dup, skipped_bad = 0, 0, 0

for subdir in sorted(os.listdir(RAW_DIR)):
    subpath = os.path.join(RAW_DIR, subdir)
    if not os.path.isdir(subpath) or subdir == 'all_unique':
        continue
    for fname in os.listdir(subpath):
        if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        fpath = os.path.join(subpath, fname)
        # Check image loads and is large enough
        img = cv2.imread(fpath)
        if img is None or img.shape[0] < 100 or img.shape[1] < 100:
            skipped_bad += 1
            continue
        # Deduplicate by MD5
        with open(fpath, 'rb') as f:
            h = hashlib.md5(f.read()).hexdigest()
        if h in seen_hashes:
            skipped_dup += 1
            continue
        seen_hashes.add(h)
        # Copy with unique name: q00_000001.jpg etc
        ext = os.path.splitext(fname)[1].lower()
        dest_name = f'{subdir}_{fname}'
        shutil.copy2(fpath, os.path.join(ALL_UNIQUE_DIR, dest_name))
        kept += 1

print(f'Kept:            {kept}')
print(f'Skipped (dup):   {skipped_dup}')
print(f'Skipped (bad):   {skipped_bad}')
print(f'\nUnique usable images: {kept}')
if kept < 300:
    print('⚠️  Below 300 — consider adding more search queries in Cell A2 and re-running.')
else:
    print('✓  Enough images to proceed to annotation.')

## Part B — Upload images to Roboflow for annotation

We upload all unique images to a Roboflow project so you can annotate them in their web UI.

### One-time setup (do this before running cell B1):
1. Go to **[app.roboflow.com](https://app.roboflow.com)** → create a free account
2. Click **"Create New Project"**
   - Project name: `robot-parts`
   - Project type: **Object Detection**
   - Annotation group: `robot`
3. Go to **Settings → API** → copy your **Private API Key**
4. Paste the key below in `ROBOFLOW_API_KEY`
5. Copy your **workspace slug** from the URL: `app.roboflow.com/<workspace-slug>/robot-parts`

# Cell B1 — Upload images to Roboflow project
!pip install -q roboflow

ROBOFLOW_API_KEY = 'YOUR_API_KEY_HERE'   # ← paste your key
MY_WORKSPACE     = 'your-workspace-slug' # ← paste your workspace slug
MY_PROJECT       = 'robot-parts'         # ← must match project name exactly

from roboflow import Roboflow
import glob

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(MY_WORKSPACE).project(MY_PROJECT)

image_files = glob.glob(os.path.join(ALL_UNIQUE_DIR, '*.jpg')) + \
              glob.glob(os.path.join(ALL_UNIQUE_DIR, '*.jpeg')) + \
              glob.glob(os.path.join(ALL_UNIQUE_DIR, '*.png'))

print(f'Uploading {len(image_files)} images to Roboflow project "{MY_PROJECT}"...')
print('This takes ~5–15 minutes depending on image count.\n')

failed = []
for i, img_path in enumerate(image_files):
    try:
        project.upload(img_path)
        if (i + 1) % 50 == 0:
            print(f'  Uploaded {i+1}/{len(image_files)}...')
    except Exception as e:
        failed.append(img_path)

print(f'\nDone. Uploaded: {len(image_files) - len(failed)}, Failed: {len(failed)}')
print(f'\n→ Now go to app.roboflow.com/{MY_WORKSPACE}/{MY_PROJECT} to annotate.')

## ⏸️  PAUSE HERE — Annotate in Roboflow Web UI

Go to **app.roboflow.com → your project → Annotate** and label all images.

### Your 5 classes (add these exactly):
| Class name | What to box |
|---|---|
| `arm` | Upper arm, forearm, or full arm of the robot |
| `leg` | Upper leg, lower leg, or full leg |
| `torso` | The main body/chest/trunk |
| `head` | The head unit |
| `sensor` | Any camera, LIDAR, depth sensor, or sensor housing |

### Annotation tips for quality:
- Draw **tight boxes** — don't include large empty background
- If a part is **more than 50% occluded**, skip it
- Label **every visible part** in each image, not just one per image
- `sensor` is often on the head — draw a separate box around it
- Skip images that are **too blurry** or show only background

### Keyboard shortcuts in Roboflow annotator:
- `B` — bounding box tool
- `1–5` — select class by number
- `A/D` — previous/next image
- `Space` — save and next

### When done annotating:
1. Click **Generate** → Version 1
2. Split: **70% train / 15% val / 15% test** (Roboflow does this automatically)
3. Preprocessing: resize to 640×640
4. No augmentations (we handle augmentation in Phase 4)
5. Click **Generate Version**
6. Come back and run Part C below

## Part C — Export annotated dataset from Roboflow → Drive

In [ ]:
# Cell C2 — Reorganize Roboflow export into project folder structure + carve calibration set
# Roboflow exports: train/images, train/labels, valid/images, valid/labels, test/...
# We need:          images/train, labels/train, images/val, labels/val, etc.

EXPORT_DIR = f'{PROJECT_DIR}/data/roboflow_export'
SPLIT_MAP = {'train': 'train', 'valid': 'val', 'test': 'test'}

for rf_split, our_split in SPLIT_MAP.items():
    for kind in ('images', 'labels'):
        src = os.path.join(EXPORT_DIR, rf_split, kind)
        dst = os.path.join(DATA_DIR, kind, our_split)
        if not os.path.exists(src):
            print(f'  Missing: {src}')
            continue
        os.makedirs(dst, exist_ok=True)
        files = os.listdir(src)
        for f in files:
            shutil.copy2(os.path.join(src, f), os.path.join(dst, f))
        print(f'  {rf_split}/{kind} → {kind}/{our_split}  ({len(files)} files)')

# Carve calibration set (100 images from train) — run once, never redo
CALIB_N = 100
train_imgs = [f for f in os.listdir(f'{DATA_DIR}/images/train')
              if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
existing_calib = [f for f in os.listdir(f'{DATA_DIR}/images/calibration')
                  if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

if existing_calib:
    print(f'\nCalibration set already exists ({len(existing_calib)} images). Skipping.')
elif len(train_imgs) < CALIB_N + 50:
    print(f'\n⚠️  Only {len(train_imgs)} train images — need at least {CALIB_N + 50} to carve calibration.')
else:
    random.seed(42)
    calib_imgs = random.sample(train_imgs, CALIB_N)
    for fname in calib_imgs:
        stem = os.path.splitext(fname)[0]
        shutil.move(f'{DATA_DIR}/images/train/{fname}',
                    f'{DATA_DIR}/images/calibration/{fname}')
        lbl = f'{DATA_DIR}/labels/train/{stem}.txt'
        if os.path.exists(lbl):
            shutil.move(lbl, f'{DATA_DIR}/labels/calibration/{stem}.txt')
    print(f'\n✓ Calibration set: {CALIB_N} images')
    print(f'✓ Train remaining:  {len(train_imgs) - CALIB_N} images')

# Summary
for split in ('train', 'val', 'test', 'calibration'):
    n = len([f for f in os.listdir(f'{DATA_DIR}/images/{split}')
             if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    print(f'  {split:12s}: {n} images')
total = sum(
    len([f for f in os.listdir(f'{DATA_DIR}/images/{s}')
         if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    for s in ('train', 'val', 'test', 'calibration')
)
print(f'  {"TOTAL":12s}: {total} images')
if total < 300:
    print('⚠️  Below 300 total — annotate more images before proceeding.')
else:
    print('✓  Dataset size OK.')

In [ ]:
# Cell C1 — Download annotated dataset from Roboflow (run after annotation is complete)
MY_VERSION = 1  # increment if you regenerated the dataset

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
dataset = rf.workspace(MY_WORKSPACE).project(MY_PROJECT).version(MY_VERSION).download(
    'yolov8',
    location=f'{PROJECT_DIR}/data/roboflow_export'
)
print('Dataset downloaded.')

## VIZ 1.A — Class distribution bar chart

In [ ]:
def count_class_distribution(labels_dir):
    counts = collections.Counter()
    if not os.path.exists(labels_dir):
        return counts
    for label_file in os.listdir(labels_dir):
        if not label_file.endswith('.txt'): continue
        with open(os.path.join(labels_dir, label_file)) as f:
            for line in f:
                line = line.strip()
                if line:
                    counts[int(line.split()[0])] += 1
    return counts

train_counts = count_class_distribution(f'{DATA_DIR}/labels/train')
bar_values = [train_counts[i] for i in range(5)]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(CLASS_NAMES, bar_values, color='steelblue', edgecolor='white')
ax.axhline(y=25, color='red', linestyle='--', label='min threshold (25)')
for bar, count in zip(bars, bar_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(count), ha='center', va='bottom', fontsize=11)
ax.set_ylabel('Annotated instances (train)')
ax.set_title('VIZ 1.A — Class distribution (training set)')
ax.legend()
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/viz1a_class_distribution.png', dpi=150)
plt.show()
print('⚠️  Any bar below red line = not enough data. Fix before training.')

## VIZ 1.B — 20 random annotated images

In [ ]:
def draw_annotations(img_path, label_path):
    img = cv2.imread(img_path)
    if img is None: return np.zeros((224,224,3), dtype=np.uint8)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    COLORS_BGR = [(255,80,80),(80,200,80),(80,80,255),(255,200,0),(200,80,255)]
    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5: continue
                cls = int(parts[0])
                cx,cy,bw,bh = float(parts[1]),float(parts[2]),float(parts[3]),float(parts[4])
                x1=int((cx-bw/2)*w); y1=int((cy-bh/2)*h)
                x2=int((cx+bw/2)*w); y2=int((cy+bh/2)*h)
                color = COLORS_BGR[cls % 5]
                cv2.rectangle(img,(x1,y1),(x2,y2),color,2)
                cv2.putText(img,CLASS_NAMES[cls],(x1,max(y1-6,0)),
                            cv2.FONT_HERSHEY_SIMPLEX,0.55,color,2)
    return img

img_dir = f'{DATA_DIR}/images/train'
lbl_dir = f'{DATA_DIR}/labels/train'
img_files = [f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))]
sample = random.sample(img_files, min(20, len(img_files)))

fig, axes = plt.subplots(4, 5, figsize=(20, 16))
for ax, fname in zip(axes.flatten(), sample):
    stem = os.path.splitext(fname)[0]
    img = draw_annotations(f'{img_dir}/{fname}', f'{lbl_dir}/{stem}.txt')
    ax.imshow(img); ax.set_title(fname[:20], fontsize=8); ax.axis('off')
for ax in axes.flatten()[len(sample):]: ax.axis('off')
plt.suptitle('VIZ 1.B — 20 random annotated training images', fontsize=14)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/viz1b_annotation_sample.png', dpi=120)
plt.show()
print('⚠️  MANUAL CHECK: tight boxes? correct class labels? No whole-image boxes?')

## VIZ 1.C — Bounding box size distribution

In [ ]:
widths, heights = [], []
for lf in os.listdir(f'{DATA_DIR}/labels/train'):
    if not lf.endswith('.txt'): continue
    with open(f'{DATA_DIR}/labels/train/{lf}') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                widths.append(float(parts[3]))
                heights.append(float(parts[4]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths, bins=30, color='steelblue', edgecolor='white')
axes[0].axvline(x=0.8, color='red', linestyle='--', label='>0.8 suspicious')
axes[0].set_xlabel('Normalized box width'); axes[0].set_title('Box width distribution')
axes[0].legend()
axes[1].hist(heights, bins=30, color='darkorange', edgecolor='white')
axes[1].axvline(x=0.8, color='red', linestyle='--', label='>0.8 suspicious')
axes[1].set_xlabel('Normalized box height'); axes[1].set_title('Box height distribution')
axes[1].legend()
plt.suptitle('VIZ 1.C — Bounding box size distribution')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/viz1c_box_sizes.png', dpi=150)
plt.show()

print(f'Total annotations: {len(widths)}')
print(f'Boxes width  > 0.8: {sum(w>0.8 for w in widths)}  ← annotation errors if high')
print(f'Boxes height > 0.8: {sum(h>0.8 for h in heights)}  ← annotation errors if high')
print(f'Boxes width  < 0.02: {sum(w<0.02 for w in widths)}  ← too small to be useful')

## Phase 1 Completion Checklist

Before moving to Phase 2, verify:

- [ ] ≥ 300 total images (train + val + test)
- [ ] All images annotated in YOLO format
- [ ] Train/val/test/calibration split done
- [ ] Calibration set: exactly 100 images carved from train
- [ ] **VIZ 1.A saved** — all class bars ≥ 25
- [ ] **VIZ 1.B saved** — boxes visually correct
- [ ] **VIZ 1.C saved** — no mass of boxes > 0.8
- [ ] 100% annotation coverage (every image has a label file)

If all checked, open `02_baseline_train.ipynb`.